# Laptop Detection: SSD & YOLO

**Four detection configurations:** custom SSD, custom YOLO, fine-tuned pretrained SSD300-VGG16, and fine-tuned pretrained YOLO11n. Uses the prior 94-image / 100-object COCO annotations.

Both custom architectures are implemented in PyTorch (`models.py`). The project includes data preparation, training, inference and AP@0.50 evaluation in the companion scripts.

## 1. Inspect the supplied dataset
Source JPG files are private and are **not included** in this archive. First copy `raw_images/set1`, `set2`, `set3` from the earlier project.

In [ ]:
!python run_experiments.py --report-dataset

## 2. Architecture and gradient smoke test
No trained model accuracy is claimed by this structural check. Custom YOLO shape for a single input image is `(1,13,13,3,6)`.

In [ ]:
!python run_experiments.py --smoke

## 3. Export common training split
`67 train / 8 val / 19 test`. YOLO labels use class 0; Torchvision labels use class 1.

In [ ]:
!python run_experiments.py --prepare

## 4. Train custom SSD from scratch
Three feature-map resolutions, convolutional backbone, classification and box-regression heads, default boxes. The weights start randomly initialized.

In [ ]:
!python run_experiments.py --experiment custom-ssd --epochs 15 --batch-size 2

## 5. Train custom YOLO-style detector from scratch
The 416×416 input becomes a 13×13 prediction grid, with three anchors per grid cell and six values per anchor: `(tx,ty,tw,th,objectness,class_logit)`.

In [ ]:
!python run_experiments.py --experiment custom-yolo --epochs 15 --batch-size 4

## 6. Fine-tune pretrained SSD300-VGG16
Pretrained COCO weights are automatically downloaded. The classifier is reduced to `background + laptop` while preserving the background and laptop channels.

In [ ]:
!python run_experiments.py --experiment pretrained-ssd --epochs 15 --batch-size 1

## 7. Fine-tune pretrained Ultralytics YOLO
Uses the eight-image validation split for model checkpoint selection; the 19-image test split remains held out.

In [ ]:
!python run_experiments.py --experiment pretrained-yolo --epochs 15 --batch-size 4

## 8. Compare all four trained models
`metrics.json` records AP@0.50, precision, recall and test sample count. **Do not insert made-up values** if a training run has not completed.

In [ ]:
!python compare_results.py

## 9. Review your results
After training, inspect `results/comparison.png` and each `results/<model>/prediction.png`. Use the same test split and report the limited sample size. Accuracy conclusions without running training on the JPGs would be invalid.

In [ ]:
from pathlib import Path
from IPython.display import display, Image
p=Path("results/comparison.png")
if p.exists():
    display(Image(filename=str(p)))
else:
    print("No comparison plot yet; finish training and run compare_results.py.")